# 11 — Responsible ML in Production

## What
Responsible ML production practice embeds fairness, accountability, and transparency directly into the deployment pipeline — not as a post-hoc audit, but as an automated gate. Fairness metrics are treated like accuracy metrics: models fail promotion if they exceed disparity thresholds.

## Why
The EU AI Act (2024), UK Equality Act, and US EEOC guidelines create legal liability for discriminatory automated decisions. Beyond compliance, models that discriminate on gender, race, or age in credit, hiring, or healthcare cause real harm. Automating the fairness check ensures it cannot be skipped under delivery pressure.

## Community context
Fairlearn (Microsoft), AI Fairness 360 (IBM), and Aequitas (University of Chicago) are the main open-source toolkits. Group fairness definitions (demographic parity, equalized odds, calibration) are mathematically incompatible with each other — you must choose which constraint fits the deployment context and document that choice explicitly.

In [ ]:
# Fairness monitor with automated deployment gate
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

def demographic_parity(y_pred, sensitive_attr, threshold=0.5):
    """P(ŷ=1|A=0) should equal P(ŷ=1|A=1)"""
    groups = np.unique(sensitive_attr)
    rates = {}
    for g in groups:
        mask = sensitive_attr == g
        rates[g] = (y_pred[mask] >= threshold).mean()
    disparity = max(rates.values()) - min(rates.values())
    return rates, disparity

def equalized_odds(y_true, y_pred, sensitive_attr, threshold=0.5):
    """TPR and FPR should be equal across groups"""
    groups = np.unique(sensitive_attr)
    tpr, fpr = {}, {}
    for g in groups:
        mask = sensitive_attr == g
        yt = y_true[mask]
        yp = (y_pred[mask] >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0,1]).ravel()
        tpr[g] = tp / (tp + fn + 1e-10)
        fpr[g] = fp / (fp + tn + 1e-10)
    tpr_disp = max(tpr.values()) - min(tpr.values())
    fpr_disp = max(fpr.values()) - min(fpr.values())
    return tpr, fpr, tpr_disp, fpr_disp

def calibration_disparity(y_true, y_pred, sensitive_attr, n_bins=5):
    """P(Y=1|ŷ=p, A=a) should equal p for all groups"""
    groups = np.unique(sensitive_attr)
    disparities = []
    bins = np.linspace(0, 1, n_bins+1)
    for g in groups:
        mask = sensitive_attr == g
        yt = y_true[mask]
        yp = y_pred[mask]
        for lo, hi in zip(bins, bins[1:]):
            bin_mask = (yp >= lo) & (yp < hi)
            if bin_mask.sum() > 5:
                mid = (lo + hi) / 2
                actual_rate = yt[bin_mask].mean()
                disparities.append(abs(actual_rate - mid))
    return np.mean(disparities) if disparities else 0.0

# Generate biased credit scoring scenario
np.random.seed(42)
n = 2000
gender = np.random.choice([0, 1], n)  # 0=male, 1=female
income = np.random.normal(50000 + gender*(-5000), 15000, n)
credit_score = np.random.normal(650 + gender*(-30), 80, n)
# Ground truth: repayment depends on income and credit score
logit = -3 + 0.00004*income + 0.004*credit_score
y_true = (np.random.rand(n) < 1/(1+np.exp(-logit))).astype(int)

# Train model (uses gender-correlated features)
X = np.column_stack([income, credit_score, gender])
model = LogisticRegression(max_iter=1000).fit(X, y_true)
y_pred = model.predict_proba(X)[:, 1]

# Run fairness audit
print('=== Fairness Audit Report ===')
rates, dp_disp = demographic_parity(y_pred, gender)
print(f'Demographic Parity:')
print(f'  Male approval rate:   {rates[0]:.3f}')
print(f'  Female approval rate: {rates[1]:.3f}')
print(f'  Disparity:            {dp_disp:.3f} {"FAIL (>0.1)" if dp_disp > 0.1 else "PASS"}')

tpr, fpr, tpr_d, fpr_d = equalized_odds(y_true, y_pred, gender)
print(f'\nEqualized Odds:')
print(f'  TPR Male={tpr[0]:.3f}  Female={tpr[1]:.3f}  Disparity={tpr_d:.3f} {"FAIL" if tpr_d > 0.1 else "PASS"}')
print(f'  FPR Male={fpr[0]:.3f}  Female={fpr[1]:.3f}  Disparity={fpr_d:.3f} {"FAIL" if fpr_d > 0.1 else "PASS"}')

cal_disp = calibration_disparity(y_true, y_pred, gender)
print(f'\nCalibration Disparity: {cal_disp:.3f} {"FAIL" if cal_disp > 0.05 else "PASS"}')

In [ ]:
# Fairness-aware deployment gate
FAIRNESS_THRESHOLDS = {
    'demographic_parity_disparity': 0.10,
    'equalized_odds_tpr_disparity': 0.10,
    'equalized_odds_fpr_disparity': 0.10,
    'calibration_disparity':        0.05,
}

def fairness_deployment_gate(y_true, y_pred, sensitive_attr):
    rates, dp = demographic_parity(y_pred, sensitive_attr)
    tpr, fpr, tpr_d, fpr_d = equalized_odds(y_true, y_pred, sensitive_attr)
    cal = calibration_disparity(y_true, y_pred, sensitive_attr)
    violations = []
    metrics = {
        'demographic_parity_disparity': dp,
        'equalized_odds_tpr_disparity': tpr_d,
        'equalized_odds_fpr_disparity': fpr_d,
        'calibration_disparity': cal,
    }
    for metric, val in metrics.items():
        threshold = FAIRNESS_THRESHOLDS[metric]
        if val > threshold:
            violations.append(f'{metric}={val:.4f} exceeds {threshold}')
    gate_passed = len(violations) == 0
    print('=== Deployment Gate ===')
    print(f'Gate passed: {gate_passed}')
    if violations:
        for v in violations:
            print(f'  VIOLATION: {v}')
        print('  -> Model BLOCKED from production. Requires bias mitigation.')
    else:
        print('  -> Model APPROVED for production deployment.')
    return gate_passed

fairness_deployment_gate(y_true, y_pred, gender)